In [0]:
import pandas as pd

data = {
    'customer_id': [101, 102, 101, 103, 102, 101],
    'name': ['John', 'Alice', 'John', 'David', 'Alice', 'John'],
    'city': ['Hyderabad', 'Mumbai', 'Bangalore', 'Delhi', 'Pune', 'Chennai'],
    'updated_date': [
        '2026-11-10',
        '2026-11-11',
        '2026-11-15',
        '2026-11-12',
        '2026-11-18',
        '2026-11-20'
    ]
}

df = pd.DataFrame(data)
print(df)

In [0]:
df["updated_date"] = pd.to_datetime(df["updated_date"])

latest_records = (df
                    .sort_values(["customer_id","updated_date"], ascending = [True, False])
                    .drop_duplicates(subset = "customer_id", keep = 'first')
                    )
print(latest_records)

In [0]:
'''
This sorts records by customer and latest update date, then keeps the first record per customer.

However, sorting millions of rows can consume substantial memory and time. Pandas processes data in memory, so make sure your machine has enough RAM.
'''

In [0]:
# pyspark code
from pyspark.sql import functions as F
from pyspark.sql.window import Window

window_spec = Window.partitionBy("customer_id").orderBy(
    F.col("updated_date").desc()
)

latest_records = (
    df.withColumn("rn", F.row_number().over(window_spec))
      .filter(F.col("rn") == 1)
      .drop("rn")
)

latest_records.show()

In [0]:
'''
Logic for pyspark: create a window by patitioning dataframe based on customer_id, and order by 
updated_at in descending order
this separates records based on customer id and keeps latest record on top

then add a column "rn" which gives row number for partitoned records, the latest record gets row number = 1
then filter rn =1 and drop rn col and show the df that contains only latest records
'''